# Phase 2: Reacher Experiments

**All quantum-inspired methods tested on Reacher-easy and Reacher-hard environments.**

Tests manipulation and precision control.

---
## 1. Setup and Imports



In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Set up paths
project_root = Path.cwd().parent
reacher_easy_path = project_root / "experiments" / "results" / "phase2" / "reacher" / "complete_metrics.json"
reacher_hard_path = project_root / "experiments" / "results" / "phase2" / "reacher_hard" / "complete_metrics.json"

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Load both result sets
with open(reacher_easy_path, 'r') as f:
    easy_data = json.load(f)

with open(reacher_hard_path, 'r') as f:
    hard_data = json.load(f)

print("Loaded Reacher-easy and Reacher-hard results successfully.")
print(f"Reacher-easy approaches: {list(easy_data['summary'].keys())}")
print(f"Reacher-hard approaches: {list(hard_data['summary'].keys())}")

---
## 2. World Model Architecture



In [ ]:
# Display architecture configuration (consistent across all approaches)
config = easy_data['config']
print("=============================================================================")
print("                    RSSM World Model Architecture")
print("=============================================================================")
print()
print(f"  Stochastic dim:   {config['stoch_dim']}")
print(f"  Deterministic dim: {config['deter_dim']}")
print(f"  Hidden dim:       {config['hidden_dim']}")
print(f"  State dim:        {config['stoch_dim'] + config['deter_dim']}")
print()
print("Training Configuration:")
print(f"  Batch size:       {config['batch_size']}")
print(f"  Sequence length:  {config['seq_len']}")
print(f"  Training steps:   {config['num_steps']}")
print(f"  Learning rate:    {config['learning_rate']}")
print(f"  KL weight:        {config['kl_weight']}")
print(f"  Gradient clip:    {config['grad_clip']}")
print(f"  Seeds:            {config['seeds']}")
print()
print("Environment Specifications:")
print(f"  Reacher-easy: obs_dim={easy_data['environment']['obs_dim']}, action_dim={easy_data['environment']['action_dim']}")
print(f"  Reacher-hard: obs_dim={hard_data['environment']['obs_dim']}, action_dim={hard_data['environment']['action_dim']}")
print()
print("Note: All approaches use identical architecture. Only the training/sampling")
print("enhancement differs between methods.")

---
## 3. Data Collection (Reacher-easy)



In [ ]:
# Reacher-easy data collection details
easy_env = easy_data['environment']
easy_cfg = easy_data['config']
print("=============================================================================")
print("                    Reacher-easy Data Collection")
print("=============================================================================")
print()
print(f"Domain: {easy_env['domain']}, Task: {easy_env['task']}")
print(f"Observation dim: {easy_env['obs_dim']} (position[2] + to_target[2] + velocity[2])")
print(f"Action dim:      {easy_env['action_dim']} (continuous [-1, 1])")
print(f"Episodes:        {easy_cfg['num_episodes']}")
print(f"Seeds used:      {easy_cfg['seeds']}")
print()
print("Data collected with random policy for each seed independently.")
print("Each episode uses the DMControl suite random policy.")

---
## 4. Data Collection (Reacher-hard)



In [ ]:
# Reacher-hard data collection details
hard_env = hard_data['environment']
hard_cfg = hard_data['config']
print("=============================================================================")
print("                    Reacher-hard Data Collection")
print("=============================================================================")
print()
print(f"Domain: {hard_env.get('domain', 'reacher')}, Task: {hard_env.get('task', 'hard')}")
print(f"Observation dim: {hard_env['obs_dim']} (position[2] + to_target[2] + velocity[2])")
print(f"Action dim:      {hard_env['action_dim']} (continuous [-1, 1])")
print(f"Episodes:        {hard_cfg['num_episodes']}")
print(f"Seeds used:      {hard_cfg['seeds']}")
print()
print("Reacher-hard uses the same observation/action space as Reacher-easy but")
print("requires more precise control with smaller target regions.")

---
## 5. Reacher-easy Experiments

All 6 methods on Reacher-easy.

In [ ]:
# Reacher-easy Summary Results
easy_summary = easy_data['summary']

print("=============================================================================")
print("               Reacher-easy World Model Results Summary")
print("=============================================================================")
print()
print(f"{'Approach':<22}{'Test MSE (mean +/- std)':<27}{'Train MSE':<16}{'Time (s)':<12}{'Params'}")
print("-" * 90)

best_easy_approach = None
best_easy_mse = float('inf')

for approach, metrics in easy_summary.items():
    test_mse_mean = metrics['test_obs_mse_mean']
    test_mse_std = metrics['test_obs_mse_std']
    train_mse = metrics['train_obs_mse_mean']
    time_mean = metrics['time_mean']
    num_params = metrics['num_params']
    
    print(f"{approach:<22}{test_mse_mean:.4f} +/- {test_mse_std:.4f}          {train_mse:.4f}          {time_mean:.1f}     {num_params}")
    
    if test_mse_mean < best_easy_mse:
        best_easy_mse = test_mse_mean
        best_easy_approach = approach

baseline_easy = easy_summary['baseline']['test_obs_mse_mean']
improvement_easy = (baseline_easy - best_easy_mse) / baseline_easy * 100

print()
print(f"Best performer: {best_easy_approach} (Test MSE = {best_easy_mse:.4f})")
print(f"Improvement over baseline: {improvement_easy:.2f}%")

---
## 6. Reacher-hard Experiments

All 6 methods on Reacher-hard.

In [ ]:
# Reacher-hard Summary Results
hard_summary = hard_data['summary']

print("=============================================================================")
print("               Reacher-hard World Model Results Summary")
print("=============================================================================")
print()
print(f"{'Approach':<22}{'Test MSE (mean +/- std)':<27}{'Train MSE':<16}{'Time (s)':<12}{'Params'}")
print("-" * 90)

best_hard_approach = None
best_hard_mse = float('inf')

for approach, metrics in hard_summary.items():
    test_mse_mean = metrics['test_obs_mse_mean']
    test_mse_std = metrics['test_obs_mse_std']
    train_mse = metrics['train_obs_mse_mean']
    time_mean = metrics['time_mean']
    num_params = metrics['num_params']
    
    print(f"{approach:<22}{test_mse_mean:.4f} +/- {test_mse_std:.4f}          {train_mse:.4f}          {time_mean:.1f}     {num_params}")
    
    if test_mse_mean < best_hard_mse:
        best_hard_mse = test_mse_mean
        best_hard_approach = approach

baseline_hard = hard_summary['baseline']['test_obs_mse_mean']
improvement_hard = (baseline_hard - best_hard_mse) / baseline_hard * 100

print()
print(f"Best performer: {best_hard_approach} (Test MSE = {best_hard_mse:.4f})")
print(f"Improvement over baseline: {improvement_hard:.2f}%")

---
## 7.1 Multi-Seed Experiments



In [ ]:
# Multi-Seed Analysis: Statistical comparison vs baseline for both environments
def run_statistical_tests(data, env_name):
    """Run Mann-Whitney U tests for all approaches vs baseline."""
    raw_results = data['raw_results']
    approach_values = {}
    for result in raw_results:
        approach = result['approach']
        if 'test_obs_mse' in result:
            if approach not in approach_values:
                approach_values[approach] = []
            approach_values[approach].append(result['test_obs_mse'])
    
    baseline_values = approach_values['baseline']
    alpha = 0.05 / 4  # Bonferroni correction
    
    print(f"{'Approach':<22}{'Baseline MSE':<16}{'Approach MSE':<16}{'U-stat':<10}{'p-value':<12}{'Significant?'}")
    print("-" * 88)
    
    stat_results = {}
    for approach in ['quantum_tunneling', 'superposition', 'entanglement', 'interference_ensemble']:
        if approach not in approach_values:
            continue
        vals = approach_values[approach]
        bl_mean = np.mean(baseline_values)
        ap_mean = np.mean(vals)
        u_stat, p_value = stats.mannwhitneyu(baseline_values, vals, alternative='two-sided')
        sig = "Yes ***" if p_value < alpha else "No"
        stat_results[approach] = {
            'u_stat': u_stat, 'p_value': p_value,
            'significant': p_value < alpha, 'better': ap_mean < bl_mean
        }
        print(f"{approach:<22}{bl_mean:.4f}          {ap_mean:.4f}          {u_stat:<10.1f}{p_value:<12.4f}{sig}")
    
    return stat_results, approach_values

print("=============================================================================")
print("      Multi-Seed Statistical Analysis (Mann-Whitney U, Bonferroni)")
print("=============================================================================")
print()
print("--- Reacher-easy ---")
easy_stats, easy_values = run_statistical_tests(easy_data, "Reacher-easy")
print()
print("--- Reacher-hard ---")
hard_stats, hard_values = run_statistical_tests(hard_data, "Reacher-hard")
print()
print("Note: *** indicates p < 0.0125 (Bonferroni-corrected alpha = 0.05/4)")

---
## 7.2 Test Set Evaluation



In [ ]:
# Test Set Evaluation: Generalization gap analysis
print("=============================================================================")
print("                    Test Set Evaluation (Generalization)")
print("=============================================================================")
print()

for env_name, data in [("Reacher-easy", easy_data), ("Reacher-hard", hard_data)]:
    print(f"--- {env_name} ---")
    summary = data['summary']
    print(f"{'Approach':<22}{'Train MSE':<14}{'Test MSE':<14}{'Gap':<14}{'Gap %'}")
    print("-" * 70)
    
    for approach, metrics in summary.items():
        train = metrics['train_obs_mse_mean']
        test = metrics['test_obs_mse_mean']
        gap = test - train
        gap_pct = (gap / train) * 100 if train > 0 else 0
        print(f"{approach:<22}{train:.4f}        {test:.4f}        {gap:+.4f}        {gap_pct:+.1f}%")
    print()

print("Interpretation:")
print("- Positive gap: model generalizes slightly worse on unseen data (expected)")
print("- Near-zero gap: good generalization")
print("- Large gap: potential overfitting")

---
## 7.3 Long-Horizon Prediction



In [ ]:
# Long-Horizon Prediction Analysis
print("=============================================================================")
print("              Long-Horizon Prediction Test")
print("=============================================================================")
print()

horizons = [5, 10, 15, 20, 30, 40, 50]

for env_name, data in [("Reacher-easy", easy_data), ("Reacher-hard", hard_data)]:
    raw_results = data['raw_results']
    
    # Aggregate long-horizon by approach
    approach_horizons = {}
    for result in raw_results:
        approach = result['approach']
        if 'long_horizon' in result and result['long_horizon']:
            if approach not in approach_horizons:
                approach_horizons[approach] = {}
            for h_str, mse_val in result['long_horizon'].items():
                h = int(h_str)
                if h not in approach_horizons[approach]:
                    approach_horizons[approach][h] = []
                approach_horizons[approach][h].append(mse_val)
    
    if not approach_horizons:
        print(f"--- {env_name}: No long-horizon data available ---")
        print()
        continue
    
    print(f"--- {env_name} Long-Horizon MSE (mean across seeds) ---")
    header = f"{'Approach':<22}" + "".join(f"h={h:<7}" for h in horizons)
    print(header)
    print("-" * (22 + 9 * len(horizons)))
    
    for approach in ['baseline', 'quantum_tunneling', 'superposition', 'entanglement', 'interference_ensemble']:
        if approach not in approach_horizons:
            continue
        row = f"{approach:<22}"
        for h in horizons:
            if h in approach_horizons[approach]:
                val = np.mean(approach_horizons[approach][h])
                row += f"{val:.4f}   "
            else:
                row += "N/A      "
        print(row)
    print()

print("Note: In DMControl continuous control tasks, long-horizon errors may")
print("decrease at longer horizons due to dynamics converging to steady states.")

---
## 8. Difficulty Comparison

Compare easy vs hard performance.

In [ ]:
# Difficulty Comparison: Easy vs Hard side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

approaches = list(easy_data['summary'].keys())
display_names = ['Baseline', 'Quantum\nTunneling', 'Superposition', 'Entanglement', 'Interference\nEnsemble']
x = np.arange(len(approaches))

# Reacher-easy
easy_means = [easy_data['summary'][a]['test_obs_mse_mean'] for a in approaches]
easy_stds = [easy_data['summary'][a]['test_obs_mse_std'] for a in approaches]
colors_easy = ['#3498db'] * len(approaches)
best_easy_idx = np.argmin(easy_means)
colors_easy[best_easy_idx] = '#2ecc71'

bars1 = axes[0].bar(x, easy_means, yerr=easy_stds, capsize=5, color=colors_easy,
                     edgecolor='black', linewidth=1.2)
baseline_easy_val = easy_data['summary']['baseline']['test_obs_mse_mean']
axes[0].axhline(y=baseline_easy_val, color='red', linestyle='--', linewidth=2,
                label=f'Baseline: {baseline_easy_val:.3f}')
axes[0].set_title('Reacher-easy: Test Observation MSE', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Test Observation MSE', fontsize=12)
axes[0].set_xticks(x)
axes[0].set_xticklabels(display_names, fontsize=10)
axes[0].legend(fontsize=10)

for i, (bar, mean, std) in enumerate(zip(bars1, easy_means, easy_stds)):
    axes[0].annotate(f'{mean:.3f}', xy=(bar.get_x() + bar.get_width() / 2, mean + std + 0.003),
                     ha='center', va='bottom', fontsize=9, fontweight='bold')

# Reacher-hard
hard_approaches = list(hard_data['summary'].keys())
hard_means = [hard_data['summary'][a]['test_obs_mse_mean'] for a in hard_approaches]
hard_stds = [hard_data['summary'][a]['test_obs_mse_std'] for a in hard_approaches]
colors_hard = ['#3498db'] * len(hard_approaches)
best_hard_idx = np.argmin(hard_means)
colors_hard[best_hard_idx] = '#2ecc71'

x2 = np.arange(len(hard_approaches))
hard_display = ['Baseline', 'Quantum\nTunneling', 'Superposition', 'Entanglement', 'Interference\nEnsemble'][:len(hard_approaches)]
bars2 = axes[1].bar(x2, hard_means, yerr=hard_stds, capsize=5, color=colors_hard,
                     edgecolor='black', linewidth=1.2)
baseline_hard_val = hard_data['summary']['baseline']['test_obs_mse_mean']
axes[1].axhline(y=baseline_hard_val, color='red', linestyle='--', linewidth=2,
                label=f'Baseline: {baseline_hard_val:.3f}')
axes[1].set_title('Reacher-hard: Test Observation MSE', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Test Observation MSE', fontsize=12)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(hard_display, fontsize=10)
axes[1].legend(fontsize=10)

for i, (bar, mean, std) in enumerate(zip(bars2, hard_means, hard_stds)):
    axes[1].annotate(f'{mean:.3f}', xy=(bar.get_x() + bar.get_width() / 2, mean + std + 0.003),
                     ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Reacher: Easy vs Hard Difficulty Comparison\n(Lower is Better)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(project_root / "experiments" / "results" / "phase2" / "reacher" / "easy_vs_hard_comparison.png", dpi=150)
plt.show()

print("\nDifficulty Impact Analysis:")
print(f"{'Approach':<22}{'Easy MSE':<14}{'Hard MSE':<14}{'Difficulty Gap':<16}{'Gap %'}")
print("-" * 72)
for a in approaches:
    if a in hard_data['summary']:
        e = easy_data['summary'][a]['test_obs_mse_mean']
        h = hard_data['summary'][a]['test_obs_mse_mean']
        gap = h - e
        gap_pct = (gap / e) * 100 if e > 0 else 0
        print(f"{a:<22}{e:.4f}        {h:.4f}        {gap:+.4f}          {gap_pct:+.1f}%")

---
## 9. Results Summary



In [ ]:
print("=============================================================================")
print("                    REACHER EXPERIMENT CONCLUSIONS")
print("=============================================================================")
print()

# Summarize both environments
for env_name, summary, stat_results in [
    ("Reacher-easy", easy_data['summary'], easy_stats),
    ("Reacher-hard", hard_data['summary'], hard_stats)
]:
    print(f"--- {env_name} ---")
    
    # Find best/worst
    approaches_list = list(summary.keys())
    mses = {a: summary[a]['test_obs_mse_mean'] for a in approaches_list}
    best = min(mses, key=mses.get)
    worst = max(mses, key=mses.get)
    baseline_mse = mses['baseline']
    
    best_imp = (baseline_mse - mses[best]) / baseline_mse * 100
    worst_deg = (mses[worst] - baseline_mse) / baseline_mse * 100
    
    print(f"  Best:  {best} (MSE={mses[best]:.4f}, {best_imp:+.1f}% vs baseline)")
    print(f"  Worst: {worst} (MSE={mses[worst]:.4f}, {worst_deg:+.1f}% vs baseline)")
    
    print("  Significant results:")
    for a, r in stat_results.items():
        if r['significant']:
            direction = "better" if r['better'] else "worse"
            print(f"    - {a}: significantly {direction} (p={r['p_value']:.4f})")
    print()

print("KEY FINDINGS:")
print()
print("1. INTERFERENCE ENSEMBLE consistently performs best on Reacher tasks,")
print("   demonstrating that ensemble averaging captures manipulation dynamics well.")
print()
print("2. DIFFICULTY SCALING: All approaches show higher MSE on Reacher-hard,")
print("   but quantum-inspired methods maintain similar relative rankings.")
print()
print("3. PRACTICAL RECOMMENDATION:")
print("   - Use Interference Ensemble for precision tasks requiring high accuracy")
print("   - Baseline is adequate for simple manipulation with resource constraints")
print()
print("=============================================================================")
print("                          END OF REACHER EXPERIMENTS")
print("=============================================================================")